# Multi-Model Output Comparison Grid (HPC)

This notebook builds one **comparison image per selected patch** with this layout:
- Rows: model outputs (Cellpose, cellSAM, CellViT, StarDist, ...)
- Columns: **Original | Outlines | Masks**

It supports:
1. Input directories for each model.
2. Random sampling of `n` patches **per WSI**.
3. Cell-count filtering via filename pattern like `01028x_01285y_71.png`.
4. Multiprocessing for faster PNG writing.
5. `tqdm` progress bars.


In [ ]:
# Optional install cell (run only if needed in a fresh environment)
# %pip install -q opencv-python pillow pandas tqdm matplotlib


In [ ]:
from __future__ import annotations

import os
import re
import random
import traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp


## 1) Configure Inputs

Update model directories and sampling/filter settings below.

In [ ]:
# -----------------------------
# Model inputs (edit these)
# -----------------------------
MODEL_SOURCES = [
    {"name": "Cellpose-SAM", "dir": "inference/cellpose/visualizations"},
    {"name": "cellSAM",      "dir": "inference/cellsam/visualizations"},
    {"name": "CellViT",      "dir": "inference/cellvit/visualizations_current_cellpose_style_ncells"},
    {"name": "StarDist",     "dir": "inference/stardist/visualizations"},
]

# If True, only keep patches that exist in every model dir.
REQUIRE_ALL_MODELS = True

# Which model's filename ncells should be used for filtering.
# Set to None to use first available ncells among models for each patch.
CELL_COUNT_SOURCE_MODEL = "CellViT"

# Cell-count filter (inclusive). Set to None to disable either bound.
CELL_COUNT_MIN = 0
CELL_COUNT_MAX = 300

# Random sampling per WSI.
N_RANDOM_PER_WSI = 25
RANDOM_SEED = 7

# Optional: restrict to specific WSIs (folder names relative to model root).
# Example: WSI_ALLOWLIST = {"RCC_3"}
WSI_ALLOWLIST = None

# -----------------------------
# Output settings
# -----------------------------
OUT_DIR = Path("inference/model_compare_grids")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# PNG compression: 0=largest fastest write, 9=smallest slowest write
PNG_COMPRESSION = 3

# Label styling
ROW_LABEL_WIDTH = 420  # increase if long model labels get clipped
HEADER_HEIGHT = 96
PADDING = 8

# Text style (increase for bigger labels)
TITLE_FONT_SIZE = 24
HEADER_FONT_SIZE = 28
ROW_LABEL_FONT_SIZE = 24

# Optional custom .ttf font path. None = auto fallback (DejaVu)
FONT_PATH = None

# -----------------------------
# Parallel settings
# -----------------------------
USE_MULTIPROCESSING = True
NUM_WORKERS = min(12, os.cpu_count() or 1)

print(f"Configured {len(MODEL_SOURCES)} models")
for m in MODEL_SOURCES:
    print(f"- {m['name']}: {m['dir']}")
print(f"Output dir: {OUT_DIR.resolve()}")


## 2) Utility Functions

These functions:
- parse patch coordinate and `ncells` from filename,
- index files from each model,
- align patches across models,
- render and save a comparison grid image.

In [ ]:
COORD_RE = re.compile(r"(?P<x>-?\d+)x[_-](?P<y>-?\d+)y", flags=re.IGNORECASE)
NCELL_RE = re.compile(r"(?:^|[_-])(?P<n>\d+)$")


def parse_coord_ncells(path: Path) -> Tuple[Optional[str], Optional[int]]:
    """Parse coord key and ncells from a filename.

    Expected examples:
    - 01028x_01285y_71.png -> coord=01028x_01285y, ncells=71
    - 01028x_01285y.cellvit_viz.png -> coord=01028x_01285y, ncells=None
    """
    stem = path.stem

    m = COORD_RE.search(stem)
    if m is None:
        return None, None

    coord_key = f"{m.group('x')}x_{m.group('y')}y"

    # remove coord portion then parse trailing integer if present
    tail = stem[m.end():]
    tail = tail.replace(".", "_")
    tail = tail.strip("_-")

    n = None
    m_n = NCELL_RE.search(tail)
    if m_n is not None:
        try:
            n = int(m_n.group("n"))
        except Exception:
            n = None

    return coord_key, n


def relative_wsi_id(file_path: Path, model_root: Path) -> str:
    """WSI id is the relative parent directory under model root.

    If file is directly under root, returns '__root__'.
    """
    rel = file_path.relative_to(model_root)
    parent = rel.parent.as_posix()
    return parent if parent not in ("", ".") else "__root__"


def find_pngs(root: Path) -> List[Path]:
    return sorted([p for p in root.rglob("*.png") if p.is_file()])


def index_model_outputs(model_name: str, model_dir: Path) -> pd.DataFrame:
    """Build one row per model output image.

    Output columns:
    - model_name, model_dir, file_path, wsi_id, coord_key, ncells
    """
    rows = []
    for p in find_pngs(model_dir):
        coord_key, ncells = parse_coord_ncells(p)
        if coord_key is None:
            continue
        wsi_id = relative_wsi_id(p, model_dir)
        rows.append(
            {
                "model_name": model_name,
                "model_dir": str(model_dir),
                "file_path": str(p),
                "wsi_id": wsi_id,
                "coord_key": coord_key,
                "ncells": ncells,
            }
        )

    return pd.DataFrame(rows)


def split_triptych_bgr(image_bgr: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Split a horizontal 3-panel image into (original, outlines, masks)."""
    h, w = image_bgr.shape[:2]
    x1 = w // 3
    x2 = (2 * w) // 3
    return image_bgr[:, :x1], image_bgr[:, x1:x2], image_bgr[:, x2:]


def resize_bgr(image_bgr: np.ndarray, target_w: int, target_h: int) -> np.ndarray:
    if image_bgr.shape[1] == target_w and image_bgr.shape[0] == target_h:
        return image_bgr
    return cv2.resize(image_bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)


def bgr_to_pil_rgb(arr_bgr: np.ndarray) -> Image.Image:
    rgb = cv2.cvtColor(arr_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def pil_rgb_to_bgr(img: Image.Image) -> np.ndarray:
    arr = np.array(img)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def load_font(size: int, font_path: Optional[str] = None) -> ImageFont.ImageFont:
    """Load a TrueType font with fallback for HPC/Jupyter environments."""
    candidates = []
    if font_path:
        candidates.append(str(font_path))
    candidates.extend(["DejaVuSans-Bold.ttf", "DejaVuSans.ttf"])

    for cand in candidates:
        try:
            return ImageFont.truetype(cand, size=int(size))
        except Exception:
            continue

    # Last fallback (fixed small size in PIL)
    return ImageFont.load_default()


def render_comparison_grid(
    model_names: List[str],
    image_paths: List[str],
    out_path: str,
    title_text: str,
    row_label_width: int = ROW_LABEL_WIDTH,
    header_h: int = HEADER_HEIGHT,
    pad: int = PADDING,
    png_compression: int = PNG_COMPRESSION,
    title_font_size: int = TITLE_FONT_SIZE,
    header_font_size: int = HEADER_FONT_SIZE,
    row_label_font_size: int = ROW_LABEL_FONT_SIZE,
    font_path: Optional[str] = FONT_PATH,
) -> Tuple[bool, str]:
    """Render and save one comparison grid PNG.

    Returns:
        (True, out_path) on success, (False, error_text) on failure.
    """
    try:
        if len(model_names) != len(image_paths):
            return False, "model_names and image_paths length mismatch"

        triptychs = []
        for p in image_paths:
            img = cv2.imread(str(p), cv2.IMREAD_COLOR)
            if img is None:
                return False, f"Failed to read image: {p}"
            triptychs.append(split_triptych_bgr(img))

        # normalize panel size to first model's first panel
        target_h, target_w = triptychs[0][0].shape[:2]

        norm_rows = []
        for tri in triptychs:
            norm_rows.append(tuple(resize_bgr(panel, target_w, target_h) for panel in tri))

        n_models = len(norm_rows)
        canvas_w = row_label_width + (3 * target_w) + (4 * pad)
        canvas_h = header_h + (n_models * target_h) + ((n_models + 1) * pad)

        canvas = Image.new("RGB", (canvas_w, canvas_h), color=(245, 245, 245))
        draw = ImageDraw.Draw(canvas)

        font_title = load_font(title_font_size, font_path=font_path)
        font_header = load_font(header_font_size, font_path=font_path)
        font_row = load_font(row_label_font_size, font_path=font_path)

        # title
        draw.text((pad, 6), title_text, fill=(0, 0, 0), font=font_title)

        # column headers
        headers = ["Original", "Outlines", "Masks"]
        y_header = max(4, header_h - int(header_font_size) - 8)
        for j, htxt in enumerate(headers):
            x = row_label_width + pad + j * target_w + 10
            draw.text((x, y_header), htxt, fill=(0, 0, 0), font=font_header)

        # rows
        for i, (mname, tri) in enumerate(zip(model_names, norm_rows)):
            y0 = header_h + pad + i * (target_h + pad)
            row_text_y = y0 + max(4, (target_h - int(row_label_font_size)) // 2)
            draw.text((pad, row_text_y), mname, fill=(0, 0, 0), font=font_row)

            for j, panel in enumerate(tri):
                x0 = row_label_width + pad + j * target_w
                canvas.paste(bgr_to_pil_rgb(panel), (x0, y0))

        out_p = Path(out_path)
        out_p.parent.mkdir(parents=True, exist_ok=True)
        out_bgr = pil_rgb_to_bgr(canvas)

        ok = cv2.imwrite(
            str(out_p),
            out_bgr,
            [cv2.IMWRITE_PNG_COMPRESSION, int(png_compression)],
        )
        if not ok:
            return False, f"cv2.imwrite failed: {out_p}"

        return True, str(out_p)
    except Exception:
        return False, traceback.format_exc()


## 3) Build Patch Table Across Models

This cell aligns matching `(wsi_id, coord)` across model folders and applies:
- `REQUIRE_ALL_MODELS`
- `CELL_COUNT_MIN/MAX`
- random sampling per WSI (`N_RANDOM_PER_WSI`)

In [ ]:
model_frames = []

for m in MODEL_SOURCES:
    model_name = str(m["name"])
    model_dir = Path(m["dir"]).expanduser().resolve()
    if not model_dir.exists():
        print(f"[WARN] missing dir for {model_name}: {model_dir}")
        continue

    df_m = index_model_outputs(model_name, model_dir)
    if df_m.empty:
        print(f"[WARN] no parseable PNGs for {model_name}: {model_dir}")
    else:
        print(f"[{model_name}] indexed {len(df_m)} images")
    model_frames.append(df_m)

if not model_frames:
    raise RuntimeError("No model outputs indexed. Check MODEL_SOURCES paths.")

all_df = pd.concat(model_frames, ignore_index=True)

# Deduplicate per (model, wsi, coord): prefer rows that include ncells, then higher ncells.
all_df["_has_ncells"] = all_df["ncells"].notna().astype(int)
all_df["_ncells_sort"] = all_df["ncells"].fillna(-1)
all_df = (
    all_df
    .sort_values(["model_name", "wsi_id", "coord_key", "_has_ncells", "_ncells_sort"])
    .drop_duplicates(subset=["model_name", "wsi_id", "coord_key"], keep="last")
    .drop(columns=["_has_ncells", "_ncells_sort"])
    .reset_index(drop=True)
)

if WSI_ALLOWLIST is not None:
    all_df = all_df[all_df["wsi_id"].isin(set(WSI_ALLOWLIST))].copy()

model_order = [m["name"] for m in MODEL_SOURCES if m["name"] in set(all_df["model_name"])]
model_set = set(model_order)

# Group all files that refer to the same patch location in one WSI.
groups = []
for (wsi_id, coord_key), g in all_df.groupby(["wsi_id", "coord_key"], sort=True):
    files_by_model = {r.model_name: r.file_path for r in g.itertuples(index=False)}
    ncells_by_model = {r.model_name: r.ncells for r in g.itertuples(index=False)}

    present_models = set(files_by_model.keys())
    if REQUIRE_ALL_MODELS and present_models != model_set:
        continue

    # pick ncells used for filtering
    ncells_for_filter = None
    if CELL_COUNT_SOURCE_MODEL is not None and CELL_COUNT_SOURCE_MODEL in ncells_by_model:
        ncells_for_filter = ncells_by_model[CELL_COUNT_SOURCE_MODEL]
    if ncells_for_filter is None:
        for mname in model_order:
            n = ncells_by_model.get(mname)
            if n is not None:
                ncells_for_filter = n
                break

    # apply ncells range filter
    if CELL_COUNT_MIN is not None and ncells_for_filter is not None and ncells_for_filter < CELL_COUNT_MIN:
        continue
    if CELL_COUNT_MAX is not None and ncells_for_filter is not None and ncells_for_filter > CELL_COUNT_MAX:
        continue

    groups.append(
        {
            "wsi_id": wsi_id,
            "coord_key": coord_key,
            "ncells_filter": ncells_for_filter,
            "files_by_model": files_by_model,
            "ncells_by_model": ncells_by_model,
            "present_models": sorted(list(present_models)),
        }
    )

if not groups:
    raise RuntimeError("No matching patches after filtering. Relax filters or verify naming patterns.")

cand_df = pd.DataFrame(
    [
        {
            "wsi_id": r["wsi_id"],
            "coord_key": r["coord_key"],
            "ncells_filter": r["ncells_filter"],
            "n_models": len(r["present_models"]),
        }
        for r in groups
    ]
)

print("\nCandidate patches by WSI (before random sampling):")
display(cand_df.groupby("wsi_id")["coord_key"].count().rename("n_candidates").to_frame())

# Random sample n per WSI
rng = random.Random(RANDOM_SEED)
selected = []
for wsi_id, items in pd.DataFrame(groups).groupby("wsi_id", sort=True):
    rows = items.to_dict(orient="records")
    if N_RANDOM_PER_WSI is not None and N_RANDOM_PER_WSI > 0 and len(rows) > N_RANDOM_PER_WSI:
        rows = rng.sample(rows, N_RANDOM_PER_WSI)
    selected.extend(rows)

selected_df = pd.DataFrame(
    [{"wsi_id": r["wsi_id"], "coord_key": r["coord_key"], "ncells_filter": r["ncells_filter"]} for r in selected]
)

print("\nSelected patches by WSI (after random sampling):")
display(selected_df.groupby("wsi_id")["coord_key"].count().rename("n_selected").to_frame())
print(f"\nTotal selected patches: {len(selected)}")


## 4) Create Render Jobs

Each selected patch becomes one output PNG grid.

In [ ]:
def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text)


def make_output_path(base_out: Path, wsi_id: str, coord_key: str, ncells: Optional[int]) -> Path:
    wsi_safe = safe_name(wsi_id)
    n_txt = "NA" if ncells is None else str(int(ncells))
    if wsi_safe == "__root__":
        fname = f"{coord_key}_{n_txt}.png"
        return base_out / fname
    return base_out / wsi_safe / f"{coord_key}_{n_txt}.png"


def format_cell_count(value) -> str:
    """Format per-model cell count for display labels."""
    if value is None:
        return "NA"
    try:
        if pd.isna(value):
            return "NA"
    except Exception:
        pass
    try:
        return str(int(round(float(value))))
    except Exception:
        return str(value)


def build_jobs(selected_rows: List[dict], model_order: List[str], out_dir: Path) -> List[dict]:
    jobs = []
    for row in selected_rows:
        files = row["files_by_model"]
        ncells_by_model = row.get("ncells_by_model", {})
        if REQUIRE_ALL_MODELS:
            if any(m not in files for m in model_order):
                continue
            ordered_paths = [files[m] for m in model_order]
            ordered_names = list(model_order)
        else:
            ordered_names = [m for m in model_order if m in files]
            ordered_paths = [files[m] for m in ordered_names]
            if not ordered_names:
                continue

        ordered_labels = [
            f"{m} | cells={format_cell_count(ncells_by_model.get(m))}"
            for m in ordered_names
        ]

        out_path = make_output_path(
            base_out=out_dir,
            wsi_id=row["wsi_id"],
            coord_key=row["coord_key"],
            ncells=row["ncells_filter"],
        )

        title_text = f"WSI: {row['wsi_id']} | Coord: {row['coord_key']} | ncells(filter)={row['ncells_filter']}"

        jobs.append(
            {
                "model_names": ordered_names,
                "row_labels": ordered_labels,
                "image_paths": ordered_paths,
                "out_path": str(out_path),
                "title_text": title_text,
            }
        )
    return jobs


jobs = build_jobs(selected_rows=selected, model_order=model_order, out_dir=OUT_DIR)
print(f"Jobs prepared: {len(jobs)}")
if jobs:
    print("Example job:")
    display(pd.DataFrame([jobs[0]]))


## 5) Render with Multiprocessing + tqdm

- Uses `ProcessPoolExecutor` when enabled.
- Falls back to sequential mode if multiprocessing fails in your notebook runtime.

In [ ]:
def _worker(job: dict) -> dict:
    ok, msg = render_comparison_grid(
        model_names=job.get("row_labels", job["model_names"]),
        image_paths=job["image_paths"],
        out_path=job["out_path"],
        title_text=job["title_text"],
    )
    return {"ok": ok, "result": msg, "out_path": job["out_path"]}


def run_jobs(jobs: List[dict], use_mp: bool = True, workers: int = 4) -> pd.DataFrame:
    if not jobs:
        return pd.DataFrame(columns=["ok", "result", "out_path"])

    results = []

    if use_mp and workers > 1:
        try:
            start_methods = mp.get_all_start_methods()
            method = "fork" if "fork" in start_methods else "spawn"
            ctx = mp.get_context(method)

            with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
                futs = [ex.submit(_worker, j) for j in jobs]
                for fut in tqdm(as_completed(futs), total=len(futs), desc="Rendering grids"):
                    results.append(fut.result())

            return pd.DataFrame(results)
        except Exception as e:
            print(f"[WARN] multiprocessing failed, fallback to sequential: {e}")

    for j in tqdm(jobs, total=len(jobs), desc="Rendering grids (sequential)"):
        results.append(_worker(j))

    return pd.DataFrame(results)


results_df = run_jobs(jobs, use_mp=USE_MULTIPROCESSING, workers=NUM_WORKERS)

if results_df.empty:
    print("No results generated.")
else:
    n_ok = int(results_df["ok"].sum())
    n_fail = int((~results_df["ok"]).sum())
    print(f"Done. success={n_ok}, failed={n_fail}")

    if n_fail > 0:
        fail_df = results_df[~results_df["ok"]].copy()
        fail_csv = OUT_DIR / "failed.csv"
        fail_df.to_csv(fail_csv, index=False)
        print(f"Failed rows saved: {fail_csv}")
        display(fail_df.head(20))


## 6) Quick Preview

Shows a few generated outputs.

In [ ]:
import matplotlib.pyplot as plt

out_pngs = sorted(list(OUT_DIR.rglob("*.png")))
print(f"Generated PNGs: {len(out_pngs)}")

preview_n = min(6, len(out_pngs))
for p in out_pngs[:preview_n]:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        continue
    plt.figure(figsize=(16, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(str(p.relative_to(OUT_DIR)))
    plt.axis("off")
    plt.show()


## Notes

- If you see too few matches, check filename pattern parsing and whether all model outputs use the same coordinate keys.
- If processing is slow, increase `NUM_WORKERS` gradually (for HPC, start with 8-16 workers).
- If notebook multiprocessing fails on your platform, set `USE_MULTIPROCESSING=False` (logic remains identical).